# Problem Framing (Business Context)

##### Fictitious business case:
A subscription-based e-commerce platform wants to predict customer churn.

##### Target variable:

- churn (1 = customer churned, 0 = retained)

##### Typical business questions supported by features:

- Who is likely to churn?

- Which customer behaviors correlate with churn?

- How do pricing, tenure, and engagement interact?

# Random Forest Requires a Different Preprocessing Strategy in Comparison to Linear Model

Random Forests are tree-based models, which implies:


| Aspect                  | Linear Models | Random Forest    |
| ----------------------- | ------------- | ---------------- |
| Scaling                 | Required      | **Not required** |
| Outliers                | Sensitive     | **Robust**       |
| Feature monotonicity    | Important     | Not required     |
| Non-linear interactions | Manual        | **Automatic**    |




Therefore (comparing to Logistic Regression in file named: `sklearn-1-data-preprocessing-banchmark-ML-LogisticR`) the following were applied:

- Scaling is removed

- Imputation remains critical

- Encoding remains necessary

- Feature engineering remains highly valuable

# Generate a Fictitious Dataset

Features include:

- Missing values

- Skewed distributions

- Mixed feature types

- Business-realistic variable

In [7]:
import numpy as np
import pandas as pd

np.random.seed(42)
n = 1000

data = pd.DataFrame({
    "customer_id": range(1, n + 1),
    "age": np.random.randint(18, 75, size=n),
    "monthly_income": np.random.lognormal(mean=8, sigma=0.5, size=n),
    "tenure_months": np.random.randint(1, 120, size=n),
    "avg_monthly_spend": np.random.gamma(shape=2, scale=50, size=n),
    "num_support_tickets": np.random.poisson(lam=2, size=n),
    "contract_type": np.random.choice(
        ["month-to-month", "annual", "biennial"], size=n, p=[0.6, 0.3, 0.1]
    ),
    "payment_method": np.random.choice(
        ["credit_card", "debit_card", "paypal", "bank_transfer"], size=n
    ),
    "region": np.random.choice(
        ["North", "South", "East", "West"], size=n
    ),
    "has_discount": np.random.choice([0, 1], size=n, p=[0.7, 0.3])
})

# Target variable with business logic
data["churn"] = (
    (data["tenure_months"] < 12).astype(int)
    | (data["num_support_tickets"] > 4).astype(int)
).astype(int)

# Inject missing values
for col in ["age", "monthly_income", "avg_monthly_spend"]:
    data.loc[data.sample(frac=0.05).index, col] = np.nan

data.head()


,customer_id,age,monthly_income,tenure_months,avg_monthly_spend,num_support_tickets,contract_type,payment_method,region,has_discount,churn
0,1,56.0,1060.270083,75,162.418321,2,annual,paypal,North,1,0
1,2,NaN,2851.042456,86,237.169358,3,month-to-month,debit_card,West,0,0
2,3,46.0,1552.722700,22,81.640444,2,annual,bank_transfer,North,0,0
3,4,NaN,4166.519335,107,102.757373,3,annual,bank_transfer,South,0,0
4,5,60.0,3580.648195,89,56.336590,2,month-to-month,debit_card,North,1,0


In [8]:
data.shape

(1000, 11)

# Feature Categorization (Critical Step)

In [9]:
target = "churn"
id_cols = ["customer_id"]

numerical_features = [
    "age",
    "monthly_income",
    "tenure_months",
    "avg_monthly_spend",
    "num_support_tickets"
]

categorical_features = [
    "contract_type",
    "payment_method",
    "region"
]

binary_features = ["has_discount"]


# Business-Driven Feature Engineering
## Ratio & Interaction Features

In [10]:
data["spend_per_month_of_tenure"] = (
    data["avg_monthly_spend"] / (data["tenure_months"] + 1)
)

data["tickets_per_month"] = (
    data["num_support_tickets"] / (data["tenure_months"] + 1)
)


__Business rationale:__

- Normalize behavior over time

- Identify “high friction” customers early


## Customer Value Segmentation

In [11]:
data["customer_value_segment"] = pd.qcut(
    data["monthly_income"],
    q=4,
    labels=["low", "mid_low", "mid_high", "high"]
)


__Business rationale:__
- Aligns with marketing segmentation and pricing strategies.


## Behavioral Flags

In [12]:
data["high_support_user"] = (data["num_support_tickets"] >= 5).astype(int)
data["short_tenure"] = (data["tenure_months"] < 12).astype(int)


# . Preprocessing Strategy (ML-Oriented)
Key Design Principles


- No data leakage


- Pipeline-based


- Reusable across models


- Scalable

# Sklearn Pipelines & Transformers

## Random Forest Requires a Different Preprocessing Strategy

Random Forests are tree-based models, which implies:


| Aspect                  | Linear Models | Random Forest    |
| ----------------------- | ------------- | ---------------- |
| Scaling                 | Required      | **Not required** |
| Outliers                | Sensitive     | **Robust**       |
| Feature monotonicity    | Important     | Not required     |
| Non-linear interactions | Manual        | **Automatic**    |



Therefore:

- __Scaling is removed__

- __Imputation remains critical__

- __Encoding remains necessary__

- Feature engineering remains __highly valuable__

In [31]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.model_selection import GridSearchCV



## Numerical Pipeline (No Scaling)

In [22]:
numerical_pipeline_rf = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])


__Reason:__
- Median is robust to outliers

- Scaling benefits linear models, SVMs, neural networks

## Categorical Pipelines
__Nominal Categorical (No Order)__

In [23]:
categorical_pipeline_rf = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])


## Ordinal Pipeline (Business Order Preserved)

In [24]:
ordinal_pipeline_rf = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ordinal", OrdinalEncoder(
        categories=[["low", "mid_low", "mid_high", "high"]]
    ))
])


## Binary Features

In [25]:
binary_pipeline_rf = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent"))
])


# ColumnTransformer Assembly

In [26]:
preprocessor_rf = ColumnTransformer(
    transformers=[
        ("num", numerical_pipeline_rf, numerical_features + [
            "spend_per_month_of_tenure",
            "tickets_per_month"
        ]),
        ("cat", categorical_pipeline_rf, categorical_features),
        ("ord", ordinal_pipeline_rf, ["customer_value_segment"]),
        ("bin", binary_pipeline_rf, binary_features + [
            "high_support_user",
            "short_tenure"
        ])
    ],
    remainder="drop"
)


 # Random Forest Pipeline

In [27]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

# Split Dataset
X = data.drop(columns=[target] + id_cols)
y = data[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

## Pipeline

In [28]:
rf_pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor_rf),
    ("model", RandomForestClassifier(
        random_state=42,
        n_jobs=-1,
        class_weight="balanced"
    ))
])


__Business rationale:__

- class_weight="balanced" addresses churn imbalance

- n_jobs=-1 maximizes computational efficiency

# Hyperparameter Grid (Production-Oriented)

The grid balances __performance, interpretability, and runtime.__

In [29]:
param_grid = {
    "model__n_estimators": [200, 500],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 10, 30],
    "model__min_samples_leaf": [1, 5, 10],
    "model__max_features": ["sqrt", "log2"]
}


# GridSearchCV Setup

In [32]:
grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1,
    verbose=2
)


In [33]:
grid_search.fit(X_train, y_train)


Fitting 5 folds for each of 108 candidates, totalling 540 fits


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessing',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='median'))]),
                                                                         ['age',
                                                                          'monthly_income',
                                                                          'tenure_months',
                                                                          'avg_monthly_spend',
                                                                          'num_support_tickets',
                                                                          'spend_per_month_of_tenure',
                                                                          'tickets_per_month']),
                                                                        ('cat',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strateg...
                                                                          'high_support_user',
                                                                          'short_tenure'])])),
                                       ('model',
                                        RandomForestClassifier(class_weight='balanced',
                                                               n_jobs=-1,
                                                               random_state=42))]),
             n_jobs=-1,
             param_grid={'model__max_depth': [None, 10, 20],
                         'model__max_features': ['sqrt', 'log2'],
                         'model__min_samples_leaf': [1, 5, 10],
                         'model__min_samples_split': [2, 10, 30],
                         'model__n_estimators': [200, 500]},
             scoring='roc_auc', verbose=2)

# Best Model Extraction

In [34]:
best_model = grid_search.best_estimator_

print("Best parameters:")
print(grid_search.best_params_)

print(f"Best CV ROC-AUC: {grid_search.best_score_:.4f}")


Best parameters:
{'model__max_depth': None, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'model__n_estimators': 200}
Best CV ROC-AUC: 1.0000


# Test Set Evaluation

In [35]:
from sklearn.metrics import roc_auc_score, classification_report


In [36]:
y_pred_proba = best_model.predict_proba(X_test)[:, 1]
y_pred = best_model.predict(X_test)

print("Test ROC-AUC:", roc_auc_score(y_test, y_pred_proba))
print(classification_report(y_test, y_pred))


Test ROC-AUC: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       178
           1       1.00      1.00      1.00        22

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      1.00      1.00       200



# . Feature Importance (Post-Preprocessing)

In [37]:
feature_names = (
    best_model.named_steps["preprocessing"]
    .get_feature_names_out()
)

importances = best_model.named_steps["model"].feature_importances_

feature_importance_df = (
    pd.DataFrame({
        "feature": feature_names,
        "importance": importances
    })
    .sort_values("importance", ascending=False)
)

feature_importance_df.head(15)


,feature,importance
2,num__tenure_months,0.259699
21,bin__short_tenure,0.248987
4,num__num_support_tickets,0.121618
6,num__tickets_per_month,0.110726
5,num__spend_per_month_of_tenure,0.102724
20,bin__high_support_user,0.092532
3,num__avg_monthly_spend,0.013689
1,num__monthly_income,0.012495
0,num__age,0.011879
18,ord__customer_value_segment,0.005505
